In [11]:
import json
import os

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator

In [12]:
pd.set_option("mode.copy_on_write", True)

In [13]:
datasets = [
    'hmof', 'hmofheat', 'cof', 'ppn',
    'zeolite', 'c3h6c3h8coremof', 'ch4n2'
]

In [49]:
pretty_labels = {
    # Models
    #'cgcnn': 'CGCNN\n(81.1K)',
    #'moformer': 'MOFormer\n(11.7M)',
    #'pmtransformer': 'PMTransformer\n(86M)',
    #'spbnet': 'SpbNet\n(153M)',
    #'intellipore': 'IntelliPore\n(0.86M)',
    'cgcnn': 'CGCNN',
    'moformer': 'MOFormer',
    'pmtransformer': 'PMTransformer',
    'spbnet': 'SpbNet',
    'intellipore': 'IntelliPore',
    # Datasets
    'hmof': 'hMOF',
    'hmofheat': 'hMOF',
    'cof': 'hCOF',
    'ppn': 'hPPN',
    'zeolite': 'hZEO',
    'ch4n2': 'CoRE\ 2019',  # Escape for latex
    'c3h6c3h8coremof': 'CoRE\ 2019',  # Escape for latex
    # hmofheat
    'H2-77K-2bar': 'H$_2$ heat of adsorption at 77 K and 2 bar (MOFs)',
    'CO2-298K-2.5bar': 'CO$_2$ heat of adsorption at 298 K and 2.5 bar (MOFs)',
    # hmof
    'H2-77-2': 'H$_2$ uptake at 77 K and 2 bar (MOFs)',
    'CO2-298-2.5': 'CO$_2$ uptake at 298 K and 2.5 bar (MOFs)',
    'CH4-298-2.5': 'CH$_4$ uptake at 298 K and 2.5 bar (MOFs)',
    'N2-298-0.9': 'N$_2$ uptake at 298 K and 0.9 bar (MOFs)',
    'Xe-273-10': 'Xe binary uptake at 273 K and 10 bar (MOFs)',
    'Kr-273-10': 'Kr binary uptake at 273 K and 10 bar (MOFs)',
    # cof
    'lowbar': 'CH$_4$ uptake at 298 K and 5.8 bar (COFs)',
    'highbar': 'CH$_4$ uptake at 298 K and 65 bar (COFs)',
    # ppn
    '1bar': 'CH$_4$ uptake at 298 K and 1 bar (PPNs)',
    '65bar': 'CH$_4$ uptake at 298 K and 65 bar (PPNs)',
    # zeolite
    'unitless_KH': 'CH$_4$ unitless Henry coefficient at 298 K (ZEOs)',
    'heat_of_adsorption': 'CH$_4$ isosteric heat of adsorption at 298 K (ZEOs)',
    # c3h6c3h8coremof
    'C3H6_loadings': 'C$_3$H$_6$ binary uptake at 298 K and 1 bar (MOFs)',
    'C3H8_loadings': 'C$_3$H$_8$ binary uptake at 298 K and 1 bar (MOFs)',
    'C3H8_C3H6_Selectivity_1bar': 'C$_3$H$_8$/C$_3$H$_6$ selectivity at 298 K and 1 bar (MOFs)',
    'C3H8_C3H6_Selectivity_infinite': 'C$_3$H$_8$/C$_3$H$_6$ infinite dillution selectivity at 298 K (MOFs)',
    'C3H6_Henry_298K': 'C$_3$H$_6$ log Henry coefficient at 298 K (MOFs)',
    'C3H8_Henry_298K': 'C$_3$H$_8$ log Henry coefficient at 298 K (MOFs)',
    'TSN_S_1Bar': r'$N_\mathrm{C_3H_8} \times \log(S_\mathrm{C_3H_8/C_3H_6})$ at 298 K and 1 bar (MOFs)',
    # ch4n2
    'ch4n2ratio-0.1bar': 'CH$_4$/N$_2$ selectivity at 298 K and 0.1 bar (MOFs)',
    'ch4n2ratio-10bar': 'CH$_4$/N$_2$ selectivity at 298 K and 10 bar (MOFs)',
    'ch4n2ratio-1bar': 'CH$_4$/N$_2$ selectivity at 298 K and 1 bar (MOFs)',
    'ch4uptake-0.1bar': 'CH$_4$ binary uptake at 298 K and 0.1 bar (MOFs)',
    'ch4uptake-10bar': 'CH$_4$ binary uptake at 298 K and 10 bar (MOFs)',
    'ch4uptake-1bar': 'CH$_4$ binary uptake at 298 K and 1 bar (MOFs)',
    'n2uptake-0.1bar': 'N$_2$ binary uptake at 298 K and 0.1 bar (MOFs)',
    'n2uptake-10bar': 'N$_2$ binary uptake at 298 K and 10 bar (MOFs)',
    'n2uptake-1bar': 'N$_2$ binary uptake at 298 K and 1 bar (MOFs)',
}

<>:19: SyntaxWarning: invalid escape sequence '\ '
<>:20: SyntaxWarning: invalid escape sequence '\ '
<>:19: SyntaxWarning: invalid escape sequence '\ '
<>:20: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_220007/1318852782.py:19: SyntaxWarning: invalid escape sequence '\ '
  'ch4n2': 'CoRE\ 2019',  # Escape for latex
/tmp/ipykernel_220007/1318852782.py:20: SyntaxWarning: invalid escape sequence '\ '
  'c3h6c3h8coremof': 'CoRE\ 2019',  # Escape for latex


In [50]:
def filter_spbnet_df(df):
    df.rename(columns={'Unnamed: 0': 'Model'}, inplace=True)
    df.dropna(inplace=True)
    df = df[~df.Metric.isin(['MSE', 'MAE (Relative Reduction)', 'MAE'])]
    df.set_index('Model', inplace=True)
    df.drop(columns='Metric', inplace=True)
    return df

In [51]:
def load_json(path):
    with open(path) as fhand:
        return json.load(fhand)

In [52]:
def plot_results(dataset, task, save=False, metric='test_R2Score'):

    train_size = 5000
    test_size = 1000
    if dataset == 'hmofheat':
        test_size = 700
    if dataset == 'ch4n2':
        train_size = 3657
        test_size = 458
    if dataset == 'c3h6c3h8coremof':
        train_size = 1354
        test_size = 170
        
    size_key = 'null' if dataset != 'hmofheat' else '5000'
    run_key = '0'

    ckpt_path = 'from_experiments_pretrain_early_pretrain_lightning_logs_version_0_checkpoints_best.ckpt-to_'
    json_path = f'../experiments/evaluate/benchmark/{dataset}/{ckpt_path}{task}/results.json'

    results = spbnet_dfs[dataset][task].to_dict()
    results['intellipore'] = float(load_json(json_path)[size_key][run_key][metric])
    
    fig, ax = plt.subplots(layout='constrained')

    # Add information about train and test sizes
    info_str = f'$\\mathbf{{{pretty_labels[dataset]}}}$\nTrain size: {train_size}\nTest size: {test_size}'
    ax.text(
        0.135, 0.84, info_str,
        transform=ax.transAxes,
        bbox={'edgecolor': 'white', 'facecolor': 'white', 'alpha': 0.7},
        fontsize='small', ha='center',
    )

    bars = ax.bar(
        [pretty_labels[k] for k in results.keys()],
        results.values(),
        linewidth=2,
        color=palette,
        edgecolor='black',
    )
    ax.bar_label(bars, fmt='%.3f', padding=3, fontweight='bold');

    ax.set_ylim(min(results.values()) - 0.05, max(results.values()) + 0.06)
    ax.yaxis.set_major_locator(MaxNLocator(6))
    
    ax.set_title(pretty_labels[task])
    ax.set_ylabel('$R^2$')

    if save:
        plt.savefig(f'figures/benchmark/{dataset}_{task}.pdf')
        
    return fig

In [53]:
# Convert excel to dataframe
xl = pd.read_excel('/home/asarikas/databases/SpbNet/41467_2026_69245_MOESM3_ESM.xlsx', sheet_name=None)
spbnet_dfs = {ds: filter_spbnet_df(xl[ds]) for ds in datasets}

In [54]:
palette = sns.color_palette('tab20')
sns.set_theme(style='darkgrid')#, palette='tab20')

In [ ]:
for ds in datasets:
    for task in [i.split('-to_')[-1] for i in os.listdir(f'../experiments/evaluate/benchmark/{ds}/')]:
        fig = plot_results(ds, task, save=True)
        #fig = plot_results(ds, task, save=False)